In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/pytorch)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
from typing import Any

import torch as tr
import torch.optim as optim
import torch.testing as testing

import import_ipynb
from approx import approx # type: ignore

In [ ]:
#
# Create a PyTorch tensor from a number or a list of numbers.
#

def T(x):
    """Returns a PyTorch tensor created from `x`."""

    if isinstance(x, tr.Tensor):
        return x
   
    return tr.tensor(x, dtype=tr.float32)


def test_T():
    assert T(1/3) == approx(0.33333)
    assert T([1/1, 1/2, 1/3]) == approx([1, 0.5, 0.33333])

In [ ]:
#
# Return a histogram of a vector if integers as a tuple of (unique_values, counts).
#

def count(vector: tr.Tensor) -> tuple[tr.Tensor, tr.Tensor]:
    """Return a histogram of a vector of integers as a tuple of (unique_values, counts)."""

    if isinstance(vector, tr.Tensor) == False:
        vector = tr.tensor(vector, dtype=tr.long)

    return vector.unique(sorted=True, return_counts=True)


def test_count():
    actual = count([1, 1, 2, 1, 2, 3, 1, 2, 3, 4, 1, 2, 3, 4, 5])
    expected = (tr.tensor([1, 2, 3, 4, 5]), tr.tensor([5, 4, 3, 2, 1]))

    assert actual[0] == approx(expected[0])
    assert actual[1] == approx(expected[1])

In [ ]:
def repeat(fn, times=10, reduction=tr.mean):
    return reduction(tr.tensor([fn() for _ in range(times)]))

In [ ]:
def test_model_sgd(model, x, y, epochs=500, lr=0.1):
    optimizer = optim.SGD(model.parameters(), lr=lr)

    for _ in range(epochs):
        optimizer.zero_grad()

        if hasattr(model, 'CUSTOM_GRAD'):
            loss = model(x, y)
        else:
            p = model(x)
            loss = model.loss(p, y)

        loss.backward()
        optimizer.step()

    with tr.no_grad():
        return tr.isclose(model.predict(x), y, atol=0.01, rtol=0.0).float().mean().item()


In [ ]:
def test_model_grad(model, x, y, epochs=500, lr=0.1):
    parameters = list(model.parameters())

    for _ in range(epochs):
        if hasattr(model, 'CUSTOM_GRAD'):
            loss = model(x, y)
        else:
            p = model(x)
            loss = model.loss(p, y)

        loss.backward()

        with tr.no_grad():
            for param in parameters:
                param -= lr * param.grad
                param.grad.zero_()

    with tr.no_grad():
        return tr.isclose(model.predict(x), y, atol=0.01, rtol=0.0).float().mean().item()


In [ ]:
if __name__ == "__main__":
    test_T()
    test_count()
